# Phase 3 — Data Preprocessing and Interaction Construction

## ASSISTments Skill Builder Dataset

### Purpose

This notebook provides reproducible evidence for the data preprocessing and interaction-construction phase of the Student Personalization Memory system.

The raw ASSISTments dataset contains repeated raw-row representations, multi-skill interactions, missing concept information, and behavioural anomalies that must be handled carefully before feature engineering.

The objectives of this phase are to:

- preserve the original raw dataset,
- construct one Base Interaction per `order_id`,
- construct one Concept Interaction per `order_id + skill_id`,
- preserve multi-skill relationships,
- consolidate multi-row opportunity representations,
- preserve interactions without known skills for general student history,
- validate processed data against the raw dataset,
- investigate behavioural anomalies before applying cleaning rules,
- and produce reproducible processed datasets for later feature engineering.

No learning-state labels, final features, state thresholds, or ML models are created in this notebook.

In [1]:
from pathlib import Path

import pandas as pd


# Locate the project root from the notebook directory
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "skill_builder_data.csv"
)

raw_df = pd.read_csv(
    RAW_DATA_PATH,
    encoding="latin1",
    low_memory=False
)

print("Raw dataset loaded successfully.")
print("Shape:", raw_df.shape)
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
print("Unique order_id:", raw_df["order_id"].nunique())
print("Unique students:", raw_df["user_id"].nunique())

Raw dataset loaded successfully.
Shape: (525534, 30)
Rows: 525534
Columns: 30
Unique order_id: 346860
Unique students: 4217


## 1. Base Interaction Construction

Data Understanding established that multiple raw rows may belong to the same underlying student interaction.

`order_id` was selected as the logical Base Interaction identifier because the important interaction-level fields were found to remain stable within each `order_id`.

The Base Interaction Dataset therefore contains exactly one row per `order_id`.

This representation will later support General Student Memory and overall student-history features.

In [2]:
base_columns = [
    "order_id",
    "user_id",
    "assignment_id",
    "assistment_id",
    "problem_id",
    "correct",
    "attempt_count",
    "ms_first_response",
    "hint_count",
    "hint_total",
    "overlap_time",
    "first_action",
    "bottom_hint",
    "tutor_mode",
    "answer_type",
    "sequence_id",
    "base_sequence_id",
    "position",
    "original",
    "student_class_id",
    "teacher_id",
    "school_id"
]

base_df = (
    raw_df[base_columns]
    .drop_duplicates(
        subset=["order_id"],
        keep="first"
    )
    .reset_index(drop=True)
    .copy()
)

print("Base Interaction Dataset")
print("Shape:", base_df.shape)
print("Rows:", len(base_df))
print(
    "Unique order_id:",
    base_df["order_id"].nunique()
)
print(
    "Duplicate order_id:",
    base_df["order_id"].duplicated().sum()
)
print(
    "Unique students:",
    base_df["user_id"].nunique()
)

Base Interaction Dataset
Shape: (346860, 22)
Rows: 346860
Unique order_id: 346860
Duplicate order_id: 0
Unique students: 4217


### 1.1 Base-Interaction Field Stability Validation

The Base Interaction Dataset keeps the first raw row for each `order_id`.

This is only valid if the selected interaction-level fields do not contain conflicting values across raw rows belonging to the same `order_id`.

Therefore, all retained Base Interaction fields are checked for within-`order_id` variation before accepting the one-row-per-`order_id` representation.

In [3]:
fields_to_check = [
    "user_id",
    "assignment_id",
    "assistment_id",
    "problem_id",
    "correct",
    "attempt_count",
    "ms_first_response",
    "hint_count",
    "hint_total",
    "overlap_time",
    "first_action",
    "bottom_hint",
    "tutor_mode",
    "answer_type",
    "sequence_id",
    "base_sequence_id",
    "position",
    "original",
    "student_class_id",
    "teacher_id",
    "school_id"
]

order_groups = raw_df.groupby("order_id")

stability_results = {}

for column in fields_to_check:
    varying_order_ids = (
        order_groups[column]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )

    stability_results[column] = varying_order_ids

stability_table = pd.DataFrame(
    {
        "field": stability_results.keys(),
        "order_ids_with_conflicting_values":
            stability_results.values()
    }
)

print("Base-interaction field stability:")
print(stability_table.to_string(index=False))

print()
print(
    "Fields containing any conflict:",
    (stability_table[
        "order_ids_with_conflicting_values"
    ] > 0).sum()
)

print(
    "Total conflicting order_id-field cases:",
    stability_table[
        "order_ids_with_conflicting_values"
    ].sum()
)

Base-interaction field stability:
            field  order_ids_with_conflicting_values
          user_id                                  0
    assignment_id                                  0
    assistment_id                                  0
       problem_id                                  0
          correct                                  0
    attempt_count                                  0
ms_first_response                                  0
       hint_count                                  0
       hint_total                                  0
     overlap_time                                  0
     first_action                                  0
      bottom_hint                                  0
       tutor_mode                                  0
      answer_type                                  0
      sequence_id                                  0
 base_sequence_id                                  0
         position                                  0
         ori

## 2. Concept Interaction Construction

A Base Interaction may be associated with more than one mathematical skill. Therefore, reducing the dataset only to one row per `order_id` would lose valid concept information.

For Concept-Based Memory, one logical interaction is represented by:

`order_id + skill_id`

Only interactions with a known `skill_id` are included in the known-skill Concept Interaction Dataset.

Some `order_id + skill_id` combinations are represented by multiple raw rows because their `opportunity` values span a consecutive range. These rows are consolidated into one logical Concept Interaction using:

- `opportunity_start` = minimum observed opportunity
- `opportunity_end` = maximum observed opportunity
- `raw_row_count` = number of raw rows represented

The behavioural values are retained from the interaction because their stability was established during Data Understanding and subsequent validation.

In [4]:
known_skill_df = (
    raw_df[
        raw_df["skill_id"].notna()
    ]
    .copy()
)

concept_df = (
    known_skill_df
    .groupby(
        ["order_id", "skill_id"],
        sort=False,
        as_index=False
    )
    .agg(
        user_id=("user_id", "first"),
        assignment_id=("assignment_id", "first"),
        assistment_id=("assistment_id", "first"),
        problem_id=("problem_id", "first"),
        skill_name=("skill_name", "first"),
        correct=("correct", "first"),
        attempt_count=("attempt_count", "first"),
        ms_first_response=("ms_first_response", "first"),
        hint_count=("hint_count", "first"),
        hint_total=("hint_total", "first"),
        overlap_time=("overlap_time", "first"),
        first_action=("first_action", "first"),
        bottom_hint=("bottom_hint", "first"),
        tutor_mode=("tutor_mode", "first"),
        answer_type=("answer_type", "first"),
        sequence_id=("sequence_id", "first"),
        base_sequence_id=("base_sequence_id", "first"),
        position=("position", "first"),
        original=("original", "first"),
        student_class_id=("student_class_id", "first"),
        teacher_id=("teacher_id", "first"),
        school_id=("school_id", "first"),
        opportunity_start=("opportunity", "min"),
        opportunity_end=("opportunity", "max"),
        raw_row_count=("opportunity", "size")
    )
    .reset_index(drop=True)
)

# Known concept IDs are whole-number identifiers.
concept_df["skill_id"] = (
    concept_df["skill_id"]
    .astype("int64")
)

print("Concept Interaction Dataset")
print("Shape:", concept_df.shape)

print(
    "Unique order_id + skill_id:",
    concept_df[
        ["order_id", "skill_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate order_id + skill_id:",
    concept_df.duplicated(
        subset=["order_id", "skill_id"]
    ).sum()
)

print(
    "Unique students:",
    concept_df["user_id"].nunique()
)

print(
    "Missing skill_id:",
    concept_df["skill_id"].isna().sum()
)

print(
    "Multi-row concept interactions:",
    (concept_df["raw_row_count"] > 1).sum()
)

print(
    "Maximum raw_row_count:",
    concept_df["raw_row_count"].max()
)

print(
    "skill_id dtype:",
    concept_df["skill_id"].dtype
)

Concept Interaction Dataset
Shape: (338001, 27)
Unique order_id + skill_id: 338001
Duplicate order_id + skill_id: 0
Unique students: 4163
Missing skill_id: 0
Multi-row concept interactions: 3227
Maximum raw_row_count: 215
skill_id dtype: int64


### 2.1 Opportunity-Range Validation

Some logical Concept Interactions are represented by multiple raw rows with consecutive `opportunity` values.

After consolidation, the resulting opportunity range must satisfy two conditions:

1. `opportunity_end` must never be smaller than `opportunity_start`.
2. The inclusive opportunity-range width must equal the number of raw rows represented by that Concept Interaction.

The expected range width is calculated as:

`opportunity_end - opportunity_start + 1`

If this equals `raw_row_count`, the raw opportunity rows form a complete consecutive range without internal gaps or duplicated opportunity values.

In [5]:
invalid_ranges = (
    concept_df["opportunity_end"]
    < concept_df["opportunity_start"]
)

expected_range_width = (
    concept_df["opportunity_end"]
    - concept_df["opportunity_start"]
    + 1
)

range_width_mismatch = (
    expected_range_width
    != concept_df["raw_row_count"]
)

single_opportunity = (
    concept_df["raw_row_count"] == 1
)

multi_opportunity = (
    concept_df["raw_row_count"] > 1
)

print(
    "Concept interactions:",
    len(concept_df)
)

print(
    "Invalid opportunity ranges:",
    invalid_ranges.sum()
)

print(
    "Range-width mismatches:",
    range_width_mismatch.sum()
)

print(
    "Single-opportunity interactions:",
    single_opportunity.sum()
)

print(
    "Multi-opportunity interactions:",
    multi_opportunity.sum()
)

print()
print("Opportunity-range width summary:")
print(
    expected_range_width
    .describe()
    .round(2)
)

Concept interactions: 338001
Invalid opportunity ranges: 0
Range-width mismatches: 0
Single-opportunity interactions: 334774
Multi-opportunity interactions: 3227

Opportunity-range width summary:
count    338001.00
mean          1.36
std           5.06
min           1.00
25%           1.00
50%           1.00
75%           1.00
max         215.00
dtype: float64


### 2.2 Multi-Skill Preservation Validation

A single Base Interaction can be associated with multiple known skills.

Therefore, Concept Interaction construction must preserve the complete set of `skill_id` values associated with every `order_id`. Losing one of these associations would produce incomplete concept histories.

For every known-skill `order_id`, the set of skills in the raw dataset is compared with the set of skills in the processed Concept Interaction Dataset.

In [6]:
raw_skill_map = (
    raw_df
    .dropna(subset=["skill_id"])
    .groupby("order_id")["skill_id"]
    .apply(
        lambda values: tuple(
            sorted(
                int(value)
                for value in set(values)
            )
        )
    )
)

concept_skill_map = (
    concept_df
    .groupby("order_id")["skill_id"]
    .apply(
        lambda values: tuple(
            sorted(set(values))
        )
    )
)

raw_only_order_ids = (
    raw_skill_map.index.difference(
        concept_skill_map.index
    )
)

concept_only_order_ids = (
    concept_skill_map.index.difference(
        raw_skill_map.index
    )
)

common_order_ids = (
    raw_skill_map.index.intersection(
        concept_skill_map.index
    )
)

skill_set_mismatches = (
    raw_skill_map.loc[common_order_ids]
    != concept_skill_map.loc[common_order_ids]
).sum()

skills_per_order = (
    concept_df
    .groupby("order_id")["skill_id"]
    .nunique()
)

print(
    "Known-skill order_ids in raw:",
    len(raw_skill_map)
)

print(
    "Known-skill order_ids in concept:",
    len(concept_skill_map)
)

print(
    "Order_ids compared:",
    len(common_order_ids)
)

print(
    "Skill-set mismatches:",
    skill_set_mismatches
)

print(
    "Raw-only order_ids:",
    len(raw_only_order_ids)
)

print(
    "Concept-only order_ids:",
    len(concept_only_order_ids)
)

print()

print(
    "Multi-skill order_ids:",
    (skills_per_order > 1).sum()
)

print(
    "Maximum skills per order_id:",
    skills_per_order.max()
)

print()
print("Number of skills per known-skill base interaction:")
print(
    skills_per_order
    .value_counts()
    .sort_index()
)

Known-skill order_ids in raw: 283105
Known-skill order_ids in concept: 283105
Order_ids compared: 283105
Skill-set mismatches: 0
Raw-only order_ids: 0
Concept-only order_ids: 0

Multi-skill order_ids: 47037
Maximum skills per order_id: 4

Number of skills per known-skill base interaction:
skill_id
1    236068
2     40857
3      4501
4      1679
Name: count, dtype: int64


### 2.3 Missing-Skill Interaction Preservation

Not every student interaction has a known `skill_id`.

Removing these interactions would discard valid general behavioural history such as correctness, attempts, hints, and response time. Therefore:

- interactions without a known skill must remain in the Base Interaction Dataset,
- they must not be artificially inserted into the known-skill Concept Interaction Dataset.

This validation checks that all no-known-skill interactions are preserved correctly.

In [7]:
raw_skill_status = (
    raw_df
    .groupby("order_id")["skill_id"]
    .apply(
        lambda values: values.notna().any()
    )
)

no_known_skill_order_ids = set(
    raw_skill_status[
        ~raw_skill_status
    ].index
)

base_order_ids = set(
    base_df["order_id"]
)

concept_order_ids = set(
    concept_df["order_id"]
)

preserved_in_base = (
    no_known_skill_order_ids
    & base_order_ids
)

incorrectly_in_concept = (
    no_known_skill_order_ids
    & concept_order_ids
)

lost_from_base = (
    no_known_skill_order_ids
    - base_order_ids
)

print(
    "Raw base interactions with no known skill:",
    len(no_known_skill_order_ids)
)

print(
    "No-known-skill interactions preserved in base:",
    len(preserved_in_base)
)

print(
    "No-known-skill interactions incorrectly in concept:",
    len(incorrectly_in_concept)
)

print(
    "No-known-skill interactions lost from base:",
    len(lost_from_base)
)

print()

print(
    "Known-skill base interactions:",
    len(base_df) - len(no_known_skill_order_ids)
)

print(
    "Total base interactions:",
    len(base_df)
)

Raw base interactions with no known skill: 63755
No-known-skill interactions preserved in base: 63755
No-known-skill interactions incorrectly in concept: 0
No-known-skill interactions lost from base: 0

Known-skill base interactions: 283105
Total base interactions: 346860


## 3. Raw-to-Processed Consistency Validation

Structural correctness alone is insufficient. The behavioural values retained in the processed datasets must also correspond to the original raw records.

Two consistency checks are performed:

1. Base Interaction values are compared against the corresponding raw `order_id`.
2. Concept Interaction values and consolidated opportunity information are independently reconstructed from the raw dataset and compared with the processed Concept Interaction Dataset.

A successful comparison demonstrates that preprocessing changed the representation of the data without silently changing the underlying learning evidence.

In [8]:
# -----------------------------
# Base Interaction consistency
# -----------------------------

base_comparison_columns = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "correct",
    "attempt_count",
    "ms_first_response",
    "hint_count",
    "hint_total"
]

raw_base_reference = (
    raw_df[base_comparison_columns]
    .drop_duplicates(
        subset=["order_id"],
        keep="first"
    )
    .sort_values("order_id")
    .reset_index(drop=True)
)

processed_base_reference = (
    base_df[base_comparison_columns]
    .sort_values("order_id")
    .reset_index(drop=True)
)

base_values_match = (
    raw_base_reference.equals(
        processed_base_reference
    )
)


# -----------------------------
# Concept Interaction consistency
# -----------------------------

raw_concept_reference = (
    raw_df[
        raw_df["skill_id"].notna()
    ]
    .groupby(
        ["order_id", "skill_id"],
        sort=False,
        as_index=False
    )
    .agg(
        user_id=("user_id", "first"),
        assignment_id=("assignment_id", "first"),
        problem_id=("problem_id", "first"),
        correct=("correct", "first"),
        attempt_count=("attempt_count", "first"),
        hint_count=("hint_count", "first"),
        hint_total=("hint_total", "first"),
        ms_first_response=(
            "ms_first_response",
            "first"
        ),
        opportunity_start=(
            "opportunity",
            "min"
        ),
        opportunity_end=(
            "opportunity",
            "max"
        ),
        raw_row_count=(
            "opportunity",
            "size"
        )
    )
)

raw_concept_reference["skill_id"] = (
    raw_concept_reference["skill_id"]
    .astype("int64")
)

concept_comparison_columns = [
    "order_id",
    "skill_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "correct",
    "attempt_count",
    "hint_count",
    "hint_total",
    "ms_first_response",
    "opportunity_start",
    "opportunity_end",
    "raw_row_count"
]

raw_concept_reference = (
    raw_concept_reference[
        concept_comparison_columns
    ]
    .sort_values(
        ["order_id", "skill_id"]
    )
    .reset_index(drop=True)
)

processed_concept_reference = (
    concept_df[
        concept_comparison_columns
    ]
    .sort_values(
        ["order_id", "skill_id"]
    )
    .reset_index(drop=True)
)

concept_values_match = (
    raw_concept_reference.equals(
        processed_concept_reference
    )
)

print(
    "Base values exactly match raw reference:",
    base_values_match
)

print(
    "Concept values exactly match raw reference:",
    concept_values_match
)

print()
print(
    "Base rows compared:",
    len(processed_base_reference)
)

print(
    "Concept rows compared:",
    len(processed_concept_reference)
)

Base values exactly match raw reference: True
Concept values exactly match raw reference: True

Base rows compared: 346860
Concept rows compared: 338001


## 4. Processed Dataset Persistence Validation

After construction and validation, the Base Interaction and Concept Interaction datasets are saved under `data/processed/`.

The saved files are reloaded from disk and checked again. This verifies that CSV serialization does not introduce duplicate interactions, missing concept identifiers, accidental index columns, or invalid opportunity ranges.

In [9]:
PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

BASE_INTERACTIONS_PATH = (
    PROCESSED_DATA_DIR
    / "base_interactions.csv"
)

CONCEPT_INTERACTIONS_PATH = (
    PROCESSED_DATA_DIR
    / "concept_interactions.csv"
)

saved_base_df = pd.read_csv(
    BASE_INTERACTIONS_PATH,
    low_memory=False
)

saved_concept_df = pd.read_csv(
    CONCEPT_INTERACTIONS_PATH,
    low_memory=False
)

print("SAVED BASE DATASET")
print("Shape:", saved_base_df.shape)

print(
    "Duplicate order_id:",
    saved_base_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "Unique students:",
    saved_base_df["user_id"].nunique()
)

print(
    "Unexpected index column:",
    any(
        str(column).startswith("Unnamed:")
        for column in saved_base_df.columns
    )
)

print()
print("SAVED CONCEPT DATASET")
print("Shape:", saved_concept_df.shape)

print(
    "Duplicate order_id + skill_id:",
    saved_concept_df.duplicated(
        subset=["order_id", "skill_id"]
    ).sum()
)

print(
    "Unique students:",
    saved_concept_df["user_id"].nunique()
)

print(
    "Missing skill_id:",
    saved_concept_df["skill_id"]
    .isna()
    .sum()
)

print(
    "Multi-row concept interactions:",
    (
        saved_concept_df["raw_row_count"] > 1
    ).sum()
)

print(
    "Maximum raw_row_count:",
    saved_concept_df["raw_row_count"].max()
)

print(
    "Invalid opportunity ranges:",
    (
        saved_concept_df["opportunity_end"]
        < saved_concept_df["opportunity_start"]
    ).sum()
)

print(
    "Unexpected index column:",
    any(
        str(column).startswith("Unnamed:")
        for column in saved_concept_df.columns
    )
)

print()
print(
    "Saved concept skill_id dtype:",
    saved_concept_df["skill_id"].dtype
)

SAVED BASE DATASET
Shape: (346860, 22)
Duplicate order_id: 0
Unique students: 4217
Unexpected index column: False

SAVED CONCEPT DATASET
Shape: (338001, 27)
Duplicate order_id + skill_id: 0
Unique students: 4163
Missing skill_id: 0
Multi-row concept interactions: 3227
Maximum raw_row_count: 215
Invalid opportunity ranges: 0
Unexpected index column: False

Saved concept skill_id dtype: int64


## 5. Behavioural Data Quality Investigation

The processed interaction datasets preserve unusual behavioural observations from the raw dataset rather than automatically deleting, clipping, or replacing them.

Before defining feature-engineering rules, important behavioural fields are investigated explicitly.

The main fields examined are:

- `attempt_count`
- `ms_first_response`
- `first_action`
- hint behaviour

At this stage, unusual values are treated as observations requiring investigation, not automatically as errors.

In [10]:
def behavioural_anomaly_summary(df):
    return {
        "attempt_count = 0":
            (df["attempt_count"] == 0).sum(),

        "attempt_count > 10":
            (df["attempt_count"] > 10).sum(),

        "attempt_count > 100":
            (df["attempt_count"] > 100).sum(),

        "maximum attempt_count":
            df["attempt_count"].max(),

        "negative response time":
            (df["ms_first_response"] < 0).sum(),

        "zero response time":
            (df["ms_first_response"] == 0).sum(),

        "response > 5 minutes":
            (df["ms_first_response"] > 300000).sum(),

        "response > 1 hour":
            (df["ms_first_response"] > 3600000).sum()
    }


base_anomalies = behavioural_anomaly_summary(
    saved_base_df
)

concept_anomalies = behavioural_anomaly_summary(
    saved_concept_df
)

anomaly_table = pd.DataFrame(
    {
        "Base Interactions": base_anomalies,
        "Concept Interactions": concept_anomalies
    }
)

print("Preserved behavioural anomalies:")
print(anomaly_table)

Preserved behavioural anomalies:
                        Base Interactions  Concept Interactions
attempt_count = 0                    7213                  9245
attempt_count > 10                   2475                  2049
attempt_count > 100                   140                   122
maximum attempt_count                3824                  3824
negative response time                  7                     6
zero response time                    310                   317
response > 5 minutes                 4292                  5023
response > 1 hour                     151                   179


### 5.1 Investigation of Zero Attempt Counts

The Base Interaction Dataset contains 7,213 interactions where `attempt_count = 0`.

These values are not automatically replaced or removed because zero attempts may represent a meaningful interaction behaviour rather than missing or corrupted data.

To investigate this, zero-attempt interactions are compared with correctness, `first_action`, hint usage, and response time. The relationship between `first_action` and zero attempts is also examined across the complete Base Interaction Dataset.

The numeric `first_action` codes are retained without assigning semantic labels because an authoritative code definition is not available in the project documentation.

In [11]:
zero_attempt_df = saved_base_df[
    saved_base_df["attempt_count"] == 0
]

print(
    "Zero-attempt base interactions:",
    len(zero_attempt_df)
)

print()
print("Correct distribution:")
print(
    zero_attempt_df["correct"]
    .value_counts()
    .sort_index()
)

print()
print("First-action distribution:")
print(
    zero_attempt_df["first_action"]
    .value_counts()
    .sort_index()
)

print()
print("Hint-count distribution:")
print(
    zero_attempt_df["hint_count"]
    .value_counts()
    .sort_index()
)

print()
print("Response-time summary:")
print(
    zero_attempt_df[
        "ms_first_response"
    ].describe().round(2)
)

print()
print("Zero attempts by first_action:")

zero_attempt_by_action = (
    saved_base_df
    .groupby("first_action")["attempt_count"]
    .apply(
        lambda values:
        (values == 0).sum()
    )
)

zero_attempt_rate_by_action = (
    saved_base_df
    .groupby("first_action")["attempt_count"]
    .apply(
        lambda values:
        (values == 0).mean() * 100
    )
    .round(2)
)

zero_attempt_action_table = pd.DataFrame(
    {
        "total_interactions":
            saved_base_df[
                "first_action"
            ].value_counts().sort_index(),

        "zero_attempt_interactions":
            zero_attempt_by_action,

        "zero_attempt_percentage":
            zero_attempt_rate_by_action
    }
)

print(zero_attempt_action_table)

Zero-attempt base interactions: 7213

Correct distribution:
correct
0    6618
1     595
Name: count, dtype: int64

First-action distribution:
first_action
0     642
1     288
2    6283
Name: count, dtype: int64

Hint-count distribution:
hint_count
0    6925
1     146
2      54
3      47
4      23
5      12
6       5
7       1
Name: count, dtype: int64

Response-time summary:
count        7213.00
mean        49280.18
std        286824.22
min             0.00
25%          2372.00
50%          9105.00
75%         39297.00
max      14476486.00
Name: ms_first_response, dtype: float64

Zero attempts by first_action:
              total_interactions  ...  zero_attempt_percentage
first_action                      ...                         
0                         309271  ...                     0.21
1                          31273  ...                     0.92
2                           6316  ...                    99.48

[3 rows x 3 columns]


### 5.2 Investigation of `first_action` Behaviour

The `first_action` field contains three numeric values: `0`, `1`, and `2`.

Because an authoritative definition of these codes is not available in the project documentation, semantic labels are not assigned to them.

Instead, their observed relationships with correctness, hint usage, attempts, and response time are analysed. This allows the field to be preserved as contextual behavioural evidence without making unsupported assumptions about the meaning of its encoded values.

In [12]:
first_action_summary = (
    saved_base_df
    .groupby("first_action")
    .agg(
        interactions=("order_id", "size"),
        accuracy=("correct", "mean"),
        average_attempts=("attempt_count", "mean"),
        average_hints=("hint_count", "mean"),
        median_response_ms=(
            "ms_first_response",
            "median"
        )
    )
    .round(3)
)

hint_usage_rate = (
    saved_base_df
    .groupby("first_action")["hint_count"]
    .apply(
        lambda values:
        (values > 0).mean() * 100
    )
    .round(2)
)

zero_attempt_rate = (
    saved_base_df
    .groupby("first_action")["attempt_count"]
    .apply(
        lambda values:
        (values == 0).mean() * 100
    )
    .round(2)
)

first_action_summary[
    "hint_usage_percentage"
] = hint_usage_rate

first_action_summary[
    "zero_attempt_percentage"
] = zero_attempt_rate

print("Behaviour by first_action:")
print(first_action_summary)

print()
print("Hint usage by first_action:")
print(
    pd.crosstab(
        saved_base_df["first_action"],
        saved_base_df["hint_count"] > 0,
        margins=True
    )
)

print()
print("Correctness by first_action:")
print(
    pd.crosstab(
        saved_base_df["first_action"],
        saved_base_df["correct"],
        margins=True
    )
)

Behaviour by first_action:
              interactions  ...  zero_attempt_percentage
first_action                ...                         
0                   309271  ...                     0.21
1                    31273  ...                     0.92
2                     6316  ...                    99.48

[3 rows x 7 columns]

Hint usage by first_action:
hint_count     False   True     All
first_action                       
0             280690  28581  309271
1                  0  31273   31273
2               6316      0    6316
All           287006  59854  346860

Correctness by first_action:
correct            0       1     All
first_action                        
0              85453  223818  309271
1              31273       0   31273
2               6316       0    6316
All           123042  223818  346860


### 5.3 Investigation of Negative Response Times

`ms_first_response` represents response-time information used later as behavioural evidence.

The Base Interaction Dataset contains a very small number of negative response-time values. Because a negative elapsed response duration is not directly interpretable as valid learning behaviour, these observations are investigated individually before defining a preprocessing rule.

The corresponding `overlap_time` is also inspected to determine whether the anomaly affects multiple time-related fields.

No rows are removed at this stage.

In [13]:
negative_response_df = (
    saved_base_df[
        saved_base_df["ms_first_response"] < 0
    ]
    .copy()
)

columns_to_show = [
    "order_id",
    "user_id",
    "correct",
    "attempt_count",
    "hint_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "tutor_mode",
    "answer_type"
]

print(
    "Negative-response interactions:",
    len(negative_response_df)
)

print()

print(
    negative_response_df[
        columns_to_show
    ].to_string(index=False)
)

print()
print(
    "Negative ms_first_response AND negative overlap_time:",
    (
        (negative_response_df["ms_first_response"] < 0)
        &
        (negative_response_df["overlap_time"] < 0)
    ).sum()
)

print(
    "Negative ms_first_response but non-negative overlap_time:",
    (
        (negative_response_df["ms_first_response"] < 0)
        &
        (negative_response_df["overlap_time"] >= 0)
    ).sum()
)

print()
print(
    "Percentage of all Base Interactions:",
    round(
        len(negative_response_df)
        / len(saved_base_df)
        * 100,
        5
    ),
    "%"
)

Negative-response interactions: 7

 order_id  user_id  correct  attempt_count  hint_count  ms_first_response  overlap_time  first_action tutor_mode answer_type
 35397005    79026        1              1           0           -7759575      -7759575             0      tutor   fill_in_1
 29732401    78895        1              1           0            -314441       -314441             0      tutor     algebra
 31272249    70746        1              1           0              -4078         -4078             0      tutor     algebra
 31480044    86708        0              6           3            -800125       -758971             0      tutor     algebra
 31480404    86708        1              1           0            -842373       -842373             0      tutor     algebra
 26392381    83240        0              2           0              -6206         24688             0      tutor     algebra
 32915077    89175        1              1           0              -6576         -6576   

### 5.4 Investigation of Zero Response Times

The Base Interaction Dataset contains interactions where `ms_first_response = 0`.

A zero response time is investigated separately from negative response times because it may represent a structural system value, an instantaneous recorded event, or unavailable timing evidence.

The relationship with `overlap_time`, `first_action`, correctness, attempts, and hints is examined before deciding whether zero response times should be used in response-time features.

No interactions are removed at this stage.

In [14]:
zero_response_df = (
    saved_base_df[
        saved_base_df["ms_first_response"] == 0
    ]
    .copy()
)

print(
    "Zero-response interactions:",
    len(zero_response_df)
)

print(
    "Percentage of Base Interactions:",
    round(
        len(zero_response_df)
        / len(saved_base_df)
        * 100,
        4
    ),
    "%"
)

print()

print("first_action distribution:")
print(
    zero_response_df["first_action"]
    .value_counts()
    .sort_index()
)

print()

print("Correct distribution:")
print(
    zero_response_df["correct"]
    .value_counts()
    .sort_index()
)

print()

print("Attempt-count summary:")
print(
    zero_response_df["attempt_count"]
    .describe()
    .round(2)
)

print()

print("Hint-count summary:")
print(
    zero_response_df["hint_count"]
    .describe()
    .round(2)
)

print()

print("overlap_time summary:")
print(
    zero_response_df["overlap_time"]
    .describe()
    .round(2)
)

print()

print(
    "Zero response AND zero overlap_time:",
    (
        zero_response_df["overlap_time"] == 0
    ).sum()
)

print(
    "Zero response but positive overlap_time:",
    (
        zero_response_df["overlap_time"] > 0
    ).sum()
)

print(
    "Zero response but negative overlap_time:",
    (
        zero_response_df["overlap_time"] < 0
    ).sum()
)

Zero-response interactions: 310
Percentage of Base Interactions: 0.0894 %

first_action distribution:
first_action
0    310
Name: count, dtype: int64

Correct distribution:
correct
0    129
1    181
Name: count, dtype: int64

Attempt-count summary:
count    310.00
mean       0.89
std        1.42
min        0.00
25%        0.00
50%        0.00
75%        2.00
max       13.00
Name: attempt_count, dtype: float64

Hint-count summary:
count    310.00
mean       0.16
std        0.67
min        0.00
25%        0.00
50%        0.00
75%        0.00
max        4.00
Name: hint_count, dtype: float64

overlap_time summary:
count       310.00
mean      18596.67
std       53523.18
min           0.00
25%           0.00
50%           0.00
75%        7134.00
max      441125.00
Name: overlap_time, dtype: float64

Zero response AND zero overlap_time: 190
Zero response but positive overlap_time: 120
Zero response but negative overlap_time: 0


### 5.5 Investigation of Extreme Positive Response Times

Positive response times may also contain unusually large values caused by inactivity, interruptions, session gaps, or other circumstances that do not represent active problem-solving time.

Therefore, positive `ms_first_response` values are examined across several duration thresholds before defining response-time features.

Large positive values are not automatically deleted. The purpose of this analysis is to determine whether raw response time should be used directly or whether robust filtering/capping is required.

In [15]:
positive_response = (
    saved_base_df.loc[
        saved_base_df["ms_first_response"] > 0,
        "ms_first_response"
    ]
)

thresholds = {
    "> 1 minute": 60_000,
    "> 2 minutes": 120_000,
    "> 5 minutes": 300_000,
    "> 10 minutes": 600_000,
    "> 30 minutes": 1_800_000,
    "> 1 hour": 3_600_000,
    "> 6 hours": 21_600_000,
    "> 12 hours": 43_200_000,
    "> 24 hours": 86_400_000
}

threshold_rows = []

for label, threshold in thresholds.items():
    count = (
        positive_response > threshold
    ).sum()

    percentage = (
        count
        / len(positive_response)
        * 100
    )

    threshold_rows.append(
        {
            "threshold": label,
            "interactions": count,
            "percentage_of_positive": round(
                percentage,
                4
            )
        }
    )

threshold_table = pd.DataFrame(
    threshold_rows
)

print(
    "Positive response-time interactions:",
    len(positive_response)
)

print()

print("Positive response-time summary (ms):")
print(
    positive_response
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
    .round(2)
)

print()
print("Extreme-response thresholds:")
print(
    threshold_table.to_string(
        index=False
    )
)

print()
print(
    "Maximum positive response time (ms):",
    positive_response.max()
)

print(
    "Maximum positive response time (hours):",
    round(
        positive_response.max()
        / 3_600_000,
        2
    )
)

Positive response-time interactions: 346543

Positive response-time summary (ms):
count      346543.00
mean        44681.80
std        339414.20
min             1.00
50%         18375.00
75%         42000.00
90%         88211.60
95%        137156.00
99%        336088.12
99.5%      505905.84
99.9%     1681315.63
max      84076920.00
Name: ms_first_response, dtype: float64

Extreme-response thresholds:
   threshold  interactions  percentage_of_positive
  > 1 minute         57934                 16.7177
 > 2 minutes         21620                  6.2388
 > 5 minutes          4292                  1.2385
> 10 minutes          1286                  0.3711
> 30 minutes           321                  0.0926
    > 1 hour           151                  0.0436
   > 6 hours            12                  0.0035
  > 12 hours             6                  0.0017
  > 24 hours             0                  0.0000

Maximum positive response time (ms): 84076920
Maximum positive response time (hours):

### 5.6 Relationship Between Response Time and Overlap Time

Extreme positive `ms_first_response` values should not be capped using an arbitrary threshold without further evidence.

The dataset also contains `overlap_time`, another time-related field. Their relationship is examined to determine whether extreme first-response values are independently anomalous or whether the same long durations are consistently represented in both timing fields.

This analysis supports the later decision on how response-time evidence should be transformed for learning-state feature engineering.

In [16]:
valid_time_df = (
    saved_base_df[
        (saved_base_df["ms_first_response"] > 0)
        &
        (saved_base_df["overlap_time"] > 0)
    ]
    .copy()
)

same_time = (
    valid_time_df["ms_first_response"]
    == valid_time_df["overlap_time"]
)

time_difference = (
    valid_time_df["overlap_time"]
    - valid_time_df["ms_first_response"]
)

print(
    "Interactions with both times positive:",
    len(valid_time_df)
)

print(
    "Exactly equal response and overlap time:",
    same_time.sum()
)

print(
    "Percentage exactly equal:",
    round(
        same_time.mean() * 100,
        2
    ),
    "%"
)

print()

print("Overlap minus first-response time summary (ms):")
print(
    time_difference
    .describe()
    .round(2)
)

print()

for threshold_ms, label in [
    (300_000, "5 minutes"),
    (600_000, "10 minutes"),
    (3_600_000, "1 hour")
]:
    extreme = valid_time_df[
        valid_time_df["ms_first_response"]
        > threshold_ms
    ]

    equal_count = (
        extreme["ms_first_response"]
        == extreme["overlap_time"]
    ).sum()

    print(f"Response > {label}:")
    print("  Interactions:", len(extreme))
    print(
        "  Same as overlap_time:",
        equal_count
    )
    print(
        "  Percentage same:",
        round(
            equal_count / len(extreme) * 100,
            2
        ) if len(extreme) else 0,
        "%"
    )

print()

largest = (
    valid_time_df[
        [
            "order_id",
            "user_id",
            "ms_first_response",
            "overlap_time",
            "correct",
            "attempt_count",
            "hint_count"
        ]
    ]
    .sort_values(
        "ms_first_response",
        ascending=False
    )
    .head(15)
)

print("15 largest positive response times:")
print(
    largest.to_string(index=False)
)

Interactions with both times positive: 346539
Exactly equal response and overlap time: 230734
Percentage exactly equal: 66.58 %

Overlap minus first-response time summary (ms):
count      346539.00
mean        12797.04
std        124700.32
min           -10.00
25%             0.00
50%             0.00
75%          3688.00
max      46520416.00
dtype: float64

Response > 5 minutes:
  Interactions: 4292
  Same as overlap_time: 2463
  Percentage same: 57.39 %
Response > 10 minutes:
  Interactions: 1286
  Same as overlap_time: 792
  Percentage same: 61.59 %
Response > 1 hour:
  Interactions: 151
  Same as overlap_time: 106
  Percentage same: 70.2 %

15 largest positive response times:
 order_id  user_id  ms_first_response  overlap_time  correct  attempt_count  hint_count
 35187187    88478           84076920      84076925        0              1           0
 32190591    79678           74563450      74563450        1              1           0
 23086044    79430           66158290      6615

### 5.7 Repeated Extreme Response-Time Values

Several very large response-time values appear more than once across different interactions.

This analysis checks whether extreme durations are unique isolated observations or repeated exact values. Repeated extreme durations may indicate characteristics of the logging/session mechanism rather than purely individual student problem-solving speed.

No values are modified during this analysis.

In [17]:
extreme_response_df = (
    saved_base_df[
        saved_base_df["ms_first_response"]
        > 3_600_000
    ]
    .copy()
)

extreme_value_counts = (
    extreme_response_df[
        "ms_first_response"
    ]
    .value_counts()
)

repeated_extreme_values = (
    extreme_value_counts[
        extreme_value_counts > 1
    ]
)

print(
    "Response-time interactions > 1 hour:",
    len(extreme_response_df)
)

print(
    "Unique >1-hour response-time values:",
    extreme_response_df[
        "ms_first_response"
    ].nunique()
)

print(
    "Exact >1-hour values occurring more than once:",
    len(repeated_extreme_values)
)

print(
    "Interactions belonging to repeated >1-hour values:",
    repeated_extreme_values.sum()
)

print()

print("Most frequent >1-hour response-time values:")
print(
    extreme_value_counts
    .head(20)
)

print()

print(
    "Students represented among >1-hour interactions:",
    extreme_response_df["user_id"].nunique()
)

print(
    "Students with more than one >1-hour interaction:",
    (
        extreme_response_df
        .groupby("user_id")
        .size()
        > 1
    ).sum()
)

Response-time interactions > 1 hour: 151
Unique >1-hour response-time values: 103
Exact >1-hour values occurring more than once: 48
Interactions belonging to repeated >1-hour values: 96

Most frequent >1-hour response-time values:
ms_first_response
5182315     2
5088594     2
16081302    2
5090217     2
12795371    2
7755304     2
3960676     2
4062125     2
7809725     2
5490277     2
14920385    2
14476486    2
10760755    2
7074046     2
44637627    2
8982586     2
8646409     2
12181064    2
5233703     2
4769552     2
Name: count, dtype: int64

Students represented among >1-hour interactions: 91
Students with more than one >1-hour interaction: 27


### 5.8 Student-Level Concentration of Extreme Response Times

Extreme response times may arise repeatedly for a small number of students or may be distributed across the population.

To distinguish these possibilities, response times above 5 minutes, 10 minutes, and 1 hour are examined at the student level.

This analysis helps determine whether extreme response times should be interpreted as general learning-speed evidence or handled robustly as potentially unreliable timing observations.

In [18]:
student_extreme_summary = (
    saved_base_df
    .groupby("user_id")
    .agg(
        interactions=("order_id", "size"),

        over_5_min=(
            "ms_first_response",
            lambda x: (x > 300_000).sum()
        ),

        over_10_min=(
            "ms_first_response",
            lambda x: (x > 600_000).sum()
        ),

        over_1_hour=(
            "ms_first_response",
            lambda x: (x > 3_600_000).sum()
        )
    )
)

print(
    "Total students:",
    len(student_extreme_summary)
)

print()

for column, label in [
    ("over_5_min", "> 5 minutes"),
    ("over_10_min", "> 10 minutes"),
    ("over_1_hour", "> 1 hour")
]:
    affected = (
        student_extreme_summary[column] > 0
    )

    repeated = (
        student_extreme_summary[column] > 1
    )

    print(label)
    print(
        "  Students with >=1:",
        affected.sum()
    )
    print(
        "  Percentage of students:",
        round(
            affected.mean() * 100,
            2
        ),
        "%"
    )
    print(
        "  Students with >1:",
        repeated.sum()
    )
    print(
        "  Maximum for one student:",
        student_extreme_summary[column].max()
    )
    print()

print(
    "Students with the most >5-minute responses:"
)

print(
    student_extreme_summary
    .sort_values(
        "over_5_min",
        ascending=False
    )
    .head(15)
    .to_string()
)

Total students: 4217

> 5 minutes
  Students with >=1: 1173
  Percentage of students: 27.82 %
  Students with >1: 589
  Maximum for one student: 71

> 10 minutes
  Students with >=1: 456
  Percentage of students: 10.81 %
  Students with >1: 184
  Maximum for one student: 31

> 1 hour
  Students with >=1: 91
  Percentage of students: 2.16 %
  Students with >1: 27
  Maximum for one student: 8

Students with the most >5-minute responses:
         interactions  over_5_min  over_10_min  over_1_hour
user_id                                                    
71881            1089          71           26            4
78915             823          57           22            3
96223             823          57           22            3
79026             811          56           31            5
78968             263          53           18            3
96233             258          52           18            3
78976             875          46           17            2
96269             587

### 5.9 Response-Time Handling Decision

The response-time investigation produced the following findings:

- 7 Base Interactions contain negative `ms_first_response` values.
- 310 Base Interactions contain zero `ms_first_response`.
- Positive response times have a strongly right-skewed distribution, with a median of approximately 18.4 seconds but a maximum of approximately 23.35 hours.
- 4,292 positive interactions exceed 5 minutes, 1,286 exceed 10 minutes, and 151 exceed 1 hour.
- Long response times occur across many students rather than being isolated to a single student.
- Many extreme `ms_first_response` values are identical to `overlap_time`.
- Exact extreme durations are also repeated across multiple interactions.

Based on this evidence, no interaction is removed solely because of its response time.

The original `ms_first_response` value is preserved in the processed interaction datasets.

For later learning-state feature engineering:

1. `ms_first_response <= 0` will be treated as unavailable/invalid response-time evidence rather than as genuine response speed.
2. Positive response times will not be used blindly as an unbounded raw mean.
3. No arbitrary upper validity cutoff is imposed during this preprocessing stage because the current evidence does not uniquely justify a specific threshold.
4. A robust response-time representation will be selected during feature engineering so that extremely long durations cannot dominate the student's learning-state estimate.

This decision preserves the original evidence while separating data preservation from later feature construction.

### 5.10 Investigation of Extreme Attempt Counts

`attempt_count` is an important behavioural signal because repeated attempts may indicate difficulty with a problem.

However, the processed data contains unusually large attempt counts. Using the raw arithmetic mean without investigation could allow a small number of extreme observations to dominate student-level or concept-level features.

The distribution and upper tail of `attempt_count` are therefore examined before defining attempt-based learning-state features.

No attempt values or interactions are modified at this stage.

In [19]:
attempts = saved_base_df["attempt_count"]

attempt_thresholds = {
    "= 0": lambda x: x == 0,
    "> 1": lambda x: x > 1,
    "> 3": lambda x: x > 3,
    "> 5": lambda x: x > 5,
    "> 10": lambda x: x > 10,
    "> 20": lambda x: x > 20,
    "> 50": lambda x: x > 50,
    "> 100": lambda x: x > 100,
    "> 500": lambda x: x > 500,
    "> 1000": lambda x: x > 1000
}

threshold_rows = []

for label, condition in attempt_thresholds.items():
    mask = condition(attempts)

    threshold_rows.append(
        {
            "threshold": label,
            "interactions": mask.sum(),
            "percentage": round(
                mask.mean() * 100,
                4
            )
        }
    )

attempt_threshold_table = pd.DataFrame(
    threshold_rows
)

print("Base Interaction attempt-count summary:")
print(
    attempts.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    ).round(2)
)

print()
print("Attempt-count thresholds:")
print(
    attempt_threshold_table.to_string(
        index=False
    )
)

print()
print(
    "Maximum attempt_count:",
    attempts.max()
)

print()

print("20 largest attempt counts:")
print(
    saved_base_df[
        [
            "order_id",
            "user_id",
            "correct",
            "attempt_count",
            "hint_count",
            "ms_first_response",
            "first_action"
        ]
    ]
    .sort_values(
        "attempt_count",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

Base Interaction attempt-count summary:
count    346860.00
mean          1.64
std          12.15
min           0.00
50%           1.00
75%           1.00
90%           2.00
95%           4.00
99%           8.00
99.5%        13.00
99.9%        46.00
max        3824.00
Name: attempt_count, dtype: float64

Attempt-count thresholds:
threshold  interactions  percentage
      = 0          7213      2.0795
      > 1         70618     20.3592
      > 3         18588      5.3589
      > 5          7375      2.1262
     > 10          2475      0.7135
     > 20           960      0.2768
     > 50           312      0.0899
    > 100           140      0.0404
    > 500            11      0.0032
   > 1000             8      0.0023

Maximum attempt_count: 3824

20 largest attempt counts:
 order_id  user_id  correct  attempt_count  hint_count  ms_first_response  first_action
 27334469    83208        0           3824           3             168730             0
 27332470    83209        0           37

### 5.11 Concentration of Extreme Attempt Counts

Extreme attempt counts may represent isolated interaction anomalies, repeated behaviour from particular students, or difficulty associated with particular problems.

To investigate this, interactions with more than 10 and more than 100 attempts are examined across students and problems.

This analysis is performed before deciding whether attempt-based learning-state features require robust transformation, capping, or alternative representations.

In [20]:
for threshold in [10, 100]:

    extreme_attempt_df = saved_base_df[
        saved_base_df["attempt_count"] > threshold
    ]

    student_counts = (
        extreme_attempt_df
        .groupby("user_id")
        .size()
    )

    problem_counts = (
        extreme_attempt_df
        .groupby("problem_id")
        .size()
    )

    print(f"ATTEMPT COUNT > {threshold}")
    print(
        "Interactions:",
        len(extreme_attempt_df)
    )

    print(
        "Students represented:",
        extreme_attempt_df["user_id"].nunique()
    )

    print(
        "Students with >1 such interaction:",
        (student_counts > 1).sum()
    )

    print(
        "Maximum such interactions for one student:",
        student_counts.max()
    )

    print(
        "Problems represented:",
        extreme_attempt_df["problem_id"].nunique()
    )

    print(
        "Problems with >1 such interaction:",
        (problem_counts > 1).sum()
    )

    print(
        "Maximum such interactions for one problem:",
        problem_counts.max()
    )

    print(
        "Incorrect interactions:",
        (extreme_attempt_df["correct"] == 0).sum()
    )

    print(
        "Correct interactions:",
        (extreme_attempt_df["correct"] == 1).sum()
    )

    print()


print("Students with most >10-attempt interactions:")

over_10_student_summary = (
    saved_base_df[
        saved_base_df["attempt_count"] > 10
    ]
    .groupby("user_id")
    .agg(
        extreme_interactions=("order_id", "size"),
        maximum_attempts=("attempt_count", "max"),
        mean_attempts=("attempt_count", "mean")
    )
    .sort_values(
        "extreme_interactions",
        ascending=False
    )
    .head(15)
)

print(
    over_10_student_summary
    .round(2)
    .to_string()
)

print()
print("Problems with most >10-attempt interactions:")

over_10_problem_summary = (
    saved_base_df[
        saved_base_df["attempt_count"] > 10
    ]
    .groupby("problem_id")
    .agg(
        extreme_interactions=("order_id", "size"),
        students=("user_id", "nunique"),
        maximum_attempts=("attempt_count", "max")
    )
    .sort_values(
        "extreme_interactions",
        ascending=False
    )
    .head(15)
)

print(
    over_10_problem_summary.to_string()
)

ATTEMPT COUNT > 10
Interactions: 2475
Students represented: 1166
Students with >1 such interaction: 495
Maximum such interactions for one student: 21
Problems represented: 1752
Problems with >1 such interaction: 505
Maximum such interactions for one problem: 12
Incorrect interactions: 2454
Correct interactions: 21

ATTEMPT COUNT > 100
Interactions: 140
Students represented: 112
Students with >1 such interaction: 17
Maximum such interactions for one student: 4
Problems represented: 130
Problems with >1 such interaction: 10
Maximum such interactions for one problem: 2
Incorrect interactions: 138
Correct interactions: 2

Students with most >10-attempt interactions:
         extreme_interactions  maximum_attempts  mean_attempts
user_id                                                       
83246                      21               103          40.62
78976                      17                82          19.29
96253                      16               253          40.19
78989         

### 5.12 Sensitivity of Attempt-Based Statistics to Extreme Values

High attempt counts are strongly associated with incorrect interactions and occur across many students and problems, suggesting that they contain useful difficulty information.

However, the maximum attempt count is several thousand times larger than the median. Therefore, raw arithmetic averages may be highly sensitive to a very small number of extreme observations.

To quantify this effect, the overall mean attempt count is recalculated after progressively restricting the upper tail. The median and multi-attempt rate are also reported because they are more robust to extreme magnitudes.

This analysis is diagnostic only. No attempt values are modified in the stored processed datasets.

In [21]:
attempt_sensitivity_rows = []

cutoffs = [
    ("No upper restriction", None),
    ("attempt_count <= 100", 100),
    ("attempt_count <= 50", 50),
    ("attempt_count <= 20", 20),
    ("attempt_count <= 10", 10),
    ("attempt_count <= 5", 5),
    ("attempt_count <= 3", 3)
]

for label, cutoff in cutoffs:

    if cutoff is None:
        subset = saved_base_df.copy()
    else:
        subset = saved_base_df[
            saved_base_df["attempt_count"] <= cutoff
        ]

    attempt_sensitivity_rows.append(
        {
            "condition": label,
            "interactions": len(subset),
            "retained_percentage": round(
                len(subset)
                / len(saved_base_df)
                * 100,
                3
            ),
            "mean_attempts": round(
                subset["attempt_count"].mean(),
                3
            ),
            "median_attempts": round(
                subset["attempt_count"].median(),
                3
            ),
            "multi_attempt_percentage": round(
                (
                    subset["attempt_count"] > 1
                ).mean() * 100,
                3
            )
        }
    )

attempt_sensitivity_table = pd.DataFrame(
    attempt_sensitivity_rows
)

print("Attempt-count sensitivity analysis:")
print(
    attempt_sensitivity_table.to_string(
        index=False
    )
)

print()

print(
    "Mean / median ratio with all observations:",
    round(
        saved_base_df["attempt_count"].mean()
        / saved_base_df["attempt_count"].median(),
        3
    )
)

print()

print("Contribution of extreme attempts to total attempt count:")

total_attempt_sum = (
    saved_base_df["attempt_count"].sum()
)

for threshold in [10, 20, 50, 100, 500, 1000]:

    extreme_attempt_sum = (
        saved_base_df.loc[
            saved_base_df["attempt_count"] > threshold,
            "attempt_count"
        ].sum()
    )

    print(
        f"attempt_count > {threshold}: "
        f"{round(extreme_attempt_sum / total_attempt_sum * 100, 2)}% "
        "of total recorded attempts"
    )

Attempt-count sensitivity analysis:
           condition  interactions  retained_percentage  mean_attempts  median_attempts  multi_attempt_percentage
No upper restriction        346860              100.000          1.639              1.0                    20.359
attempt_count <= 100        346720               99.960          1.512              1.0                    20.327
 attempt_count <= 50        346548               99.910          1.479              1.0                    20.288
 attempt_count <= 20        345900               99.723          1.426              1.0                    20.138
 attempt_count <= 10        344385               99.286          1.369              1.0                    19.787
  attempt_count <= 5        339485               97.874          1.285              1.0                    18.629
  attempt_count <= 3        328272               94.641          1.181              1.0                    15.850

Mean / median ratio with all observations: 1.639

C

### 5.13 Attempt-Count Handling Decision

The attempt-count investigation produced the following findings:

- The median `attempt_count` is 1, while the maximum is 3,824.
- 20.36% of Base Interactions contain more than one attempt.
- Only 0.71% of interactions contain more than 10 attempts.
- Only 0.04% contain more than 100 attempts.
- Extreme attempt counts occur across many students and many different problems rather than being isolated to one student or problem.
- Interactions with more than 10 attempts are overwhelmingly incorrect, indicating that high attempt counts contain meaningful evidence of learning difficulty.
- Despite their rarity, interactions with more than 10 attempts contribute approximately 17.03% of the total recorded attempt count.
- Interactions above 100 attempts alone contribute approximately 7.74% of all recorded attempts.
- Removing progressively larger attempt counts substantially changes the arithmetic mean, while the median remains 1.

Based on this evidence, no interaction is removed solely because of a high `attempt_count`, and the original attempt values are preserved in the processed interaction datasets.

For later learning-state feature engineering:

1. `attempt_count = 0` will be preserved because it is strongly associated with a structured `first_action` pattern and cannot be assumed to represent missing data.
2. High attempt counts will be preserved as potential evidence of student difficulty.
3. Raw unbounded mean attempt count will not be relied upon as the only attempt-based feature because it is sensitive to the extreme upper tail.
4. Robust attempt features such as median attempt count, multi-attempt rate, threshold-based indicators, or an appropriately transformed/capped representation will be evaluated during feature engineering.
5. Any eventual transformation or cap will be applied to derived features rather than altering the original stored interaction evidence.

This approach preserves meaningful difficulty information while preventing a small number of extreme attempt counts from dominating the student's learning-state representation.

### 5.14 Investigation of Hint Variables

Hint behaviour is an important signal for identifying whether a student requires support.

The dataset contains two related fields:

- `hint_count`
- `hint_total`

Before constructing hint-based learning-state features, their distributions and relationship must be examined.

The purpose of this analysis is to determine whether the two fields represent redundant information or different aspects of hint behaviour, and whether derived measures such as hint usage rate can be constructed safely.

No hint values are modified during this investigation.

In [22]:
print("hint_count summary:")
print(
    saved_base_df["hint_count"]
    .describe()
    .round(3)
)

print()
print("hint_total summary:")
print(
    saved_base_df["hint_total"]
    .describe()
    .round(3)
)

print()
print("hint_count distribution:")
print(
    saved_base_df["hint_count"]
    .value_counts()
    .sort_index()
)

print()
print("hint_total distribution:")
print(
    saved_base_df["hint_total"]
    .value_counts()
    .sort_index()
)

print()

print(
    "Interactions with hint_count > 0:",
    (saved_base_df["hint_count"] > 0).sum()
)

print(
    "Percentage using at least one hint:",
    round(
        (saved_base_df["hint_count"] > 0).mean()
        * 100,
        2
    ),
    "%"
)

print()

print(
    "hint_count > hint_total:",
    (
        saved_base_df["hint_count"]
        > saved_base_df["hint_total"]
    ).sum()
)

print(
    "hint_count = hint_total:",
    (
        saved_base_df["hint_count"]
        == saved_base_df["hint_total"]
    ).sum()
)

print(
    "hint_count < hint_total:",
    (
        saved_base_df["hint_count"]
        < saved_base_df["hint_total"]
    ).sum()
)

print()

print(
    "Correlation between hint_count and hint_total:",
    round(
        saved_base_df[
            ["hint_count", "hint_total"]
        ].corr().iloc[0, 1],
        3
    )
)

print()

print("hint_count × hint_total table:")
print(
    pd.crosstab(
        saved_base_df["hint_count"],
        saved_base_df["hint_total"]
    )
)

hint_count summary:
count    346860.00
mean          0.51
std           1.22
min           0.00
25%           0.00
50%           0.00
75%           0.00
max          10.00
Name: hint_count, dtype: float64

hint_total summary:
count    346860.000
mean          2.381
std           1.781
min           0.000
25%           0.000
50%           3.000
75%           4.000
max          10.000
Name: hint_total, dtype: float64

hint_count distribution:
hint_count
0     287006
1       6875
2      13742
3      21458
4      12672
5       3627
6       1106
7        373
10         1
Name: count, dtype: int64

hint_total distribution:
hint_total
0     101551
1       4864
2      45618
3      92885
4      67858
5      27067
6       5175
7       1572
8        264
10         6
Name: count, dtype: int64

Interactions with hint_count > 0: 59854
Percentage using at least one hint: 17.26 %

hint_count > hint_total: 0
hint_count = hint_total: 144234
hint_count < hint_total: 202626

Correlation between hint_count

### 5.15 Investigation of Zero Available Hints

A normalized hint-usage measure such as `hint_count / hint_total` is potentially useful because different problems may provide different numbers of available hints.

However, `hint_total = 0` prevents direct calculation of this ratio and may represent interactions where no hints were available.

Before constructing a hint-usage rate, interactions with zero available hints are examined separately.

No assumption is made that `hint_total = 0` means that the student chose not to use hints. The behaviour of these interactions is investigated first.

In [23]:
zero_hint_total = (
    saved_base_df[
        saved_base_df["hint_total"] == 0
    ]
    .copy()
)

positive_hint_total = (
    saved_base_df[
        saved_base_df["hint_total"] > 0
    ]
    .copy()
)

print(
    "Interactions with hint_total = 0:",
    len(zero_hint_total)
)

print(
    "Percentage of Base Interactions:",
    round(
        len(zero_hint_total)
        / len(saved_base_df)
        * 100,
        2
    ),
    "%"
)

print()

print(
    "hint_count > 0 when hint_total = 0:",
    (zero_hint_total["hint_count"] > 0).sum()
)

print()

print("Correctness when hint_total = 0:")
print(
    zero_hint_total["correct"]
    .value_counts()
    .sort_index()
)

print()

print("first_action when hint_total = 0:")
print(
    zero_hint_total["first_action"]
    .value_counts()
    .sort_index()
)

print()

print("Behaviour comparison:")

comparison = pd.DataFrame(
    {
        "hint_total = 0": {
            "interactions": len(zero_hint_total),
            "accuracy":
                zero_hint_total["correct"].mean(),
            "avg_attempts":
                zero_hint_total["attempt_count"].mean(),
            "multi_attempt_rate":
                (zero_hint_total["attempt_count"] > 1).mean(),
            "avg_response_ms":
                zero_hint_total.loc[
                    zero_hint_total["ms_first_response"] > 0,
                    "ms_first_response"
                ].mean()
        },

        "hint_total > 0": {
            "interactions": len(positive_hint_total),
            "accuracy":
                positive_hint_total["correct"].mean(),
            "avg_attempts":
                positive_hint_total["attempt_count"].mean(),
            "multi_attempt_rate":
                (positive_hint_total["attempt_count"] > 1).mean(),
            "avg_response_ms":
                positive_hint_total.loc[
                    positive_hint_total["ms_first_response"] > 0,
                    "ms_first_response"
                ].mean()
        }
    }
).T

print(
    comparison.round(3).to_string()
)

print()

print(
    "Interactions eligible for hint_count / hint_total:",
    len(positive_hint_total)
)

print(
    "Percentage eligible:",
    round(
        len(positive_hint_total)
        / len(saved_base_df)
        * 100,
        2
    ),
    "%"
)

Interactions with hint_total = 0: 101551
Percentage of Base Interactions: 29.28 %

hint_count > 0 when hint_total = 0: 0

Correctness when hint_total = 0:
correct
0    29362
1    72189
Name: count, dtype: int64

first_action when hint_total = 0:
first_action
0    95511
2     6040
Name: count, dtype: int64

Behaviour comparison:
                interactions  accuracy  avg_attempts  multi_attempt_rate  avg_response_ms
hint_total = 0      101551.0     0.711         1.023               0.034        52561.393
hint_total > 0      245309.0     0.618         1.894               0.274        41417.154

Interactions eligible for hint_count / hint_total: 245309
Percentage eligible: 70.72 %


### 5.16 Normalized Hint-Usage Rate

`hint_count` alone does not account for differences in the number of hints available for different interactions.

For interactions where `hint_total > 0`, a normalized hint-usage rate can be calculated as:

`hint_usage_rate = hint_count / hint_total`

Interactions where `hint_total = 0` are not assigned a zero usage rate because zero would incorrectly imply that hints were available but unused. Instead, their normalized hint-usage rate is treated as unavailable.

The relationship between normalized hint usage and correctness is examined to determine whether this measure is useful for later learning-state feature engineering.

In [24]:
hint_analysis_df = (
    saved_base_df[
        saved_base_df["hint_total"] > 0
    ]
    .copy()
)

hint_analysis_df["hint_usage_rate"] = (
    hint_analysis_df["hint_count"]
    / hint_analysis_df["hint_total"]
)

print(
    "Interactions with measurable hint usage rate:",
    len(hint_analysis_df)
)

print()

print("Hint-usage-rate summary:")
print(
    hint_analysis_df["hint_usage_rate"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(3)
)

print()

print(
    "No hints used although hints available:",
    (hint_analysis_df["hint_usage_rate"] == 0).sum()
)

print(
    "Some but not all hints used:",
    (
        (hint_analysis_df["hint_usage_rate"] > 0)
        &
        (hint_analysis_df["hint_usage_rate"] < 1)
    ).sum()
)

print(
    "All available hints used:",
    (hint_analysis_df["hint_usage_rate"] == 1).sum()
)

print()

print("Behaviour by hint usage:")

hint_analysis_df["hint_usage_group"] = pd.cut(
    hint_analysis_df["hint_usage_rate"],
    bins=[-0.001, 0, 0.5, 0.999999, 1.0],
    labels=[
        "No hints used",
        "Up to 50% used",
        "More than 50% but not all",
        "All hints used"
    ]
)

hint_group_summary = (
    hint_analysis_df
    .groupby(
        "hint_usage_group",
        observed=True
    )
    .agg(
        interactions=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_attempts=("attempt_count", "mean")
    )
    .round(3)
)

print(hint_group_summary)

print()

print(
    "Correlation between hint usage rate and correctness:",
    round(
        hint_analysis_df[
            ["hint_usage_rate", "correct"]
        ].corr().iloc[0, 1],
        3
    )
)

Interactions with measurable hint usage rate: 245309

Hint-usage-rate summary:
count    245309.000
mean          0.212
std           0.391
min           0.000
25%           0.000
50%           0.000
75%           0.000
90%           1.000
95%           1.000
99%           1.000
max           1.000
Name: hint_usage_rate, dtype: float64

No hints used although hints available: 185455
Some but not all hints used: 17171
All available hints used: 42683

Behaviour by hint usage:
                           interactions  accuracy  avg_attempts
hint_usage_group                                              
No hints used                    185455     0.818         1.453
Up to 50% used                     8261     0.000         3.310
More than 50% but not all          8910     0.000         3.396
All hints used                    42683     0.000         3.219

Correlation between hint usage rate and correctness: -0.691


### 5.17 Hint-Feature Handling Decision

The hint-variable investigation produced the following findings:

- 59,854 Base Interactions (17.26%) contain at least one used hint.
- `hint_count` never exceeds `hint_total`.
- 101,551 interactions (29.28%) have `hint_total = 0`, and none of these interactions have positive `hint_count`.
- Therefore, `hint_count / hint_total` cannot be meaningfully calculated for all interactions.
- 245,309 interactions (70.72%) have `hint_total > 0` and support calculation of a normalized hint-usage rate.
- Among these interactions, higher hint usage is strongly associated with lower recorded correctness (`r = -0.691`).
- Interactions with positive hint usage have recorded correctness of zero in this dataset. This association is treated as a property of the observed interaction/logging structure and is not interpreted as evidence that requesting hints causes incorrect learning outcomes.

Based on this evidence:

1. The original `hint_count` and `hint_total` values will be preserved.
2. `hint_count` will remain available as a direct behavioural signal.
3. A normalized `hint_usage_rate = hint_count / hint_total` may be used when `hint_total > 0`.
4. When `hint_total = 0`, normalized hint usage will be treated as unavailable rather than assigning a false rate of zero.
5. The system should distinguish between "no measurable hint opportunity" and "hints were available but none were used."
6. Hint behaviour may later contribute to the student's support/difficulty representation, but its relationship with correctness will not be interpreted causally.

The exact aggregation of hint behaviour across recent student history will be defined during feature engineering.

## 6. Concept Metadata Quality

### 6.1 Missing Skill-Name Investigation

The Concept Interaction Dataset requires a reliable concept identifier.

`skill_id` is complete for all Concept Interactions, but some interactions have a missing `skill_name`. Before deciding how to handle these missing names, the relationship between `skill_id` and `skill_name` is examined.

The analysis checks:

- how many Concept Interactions have a missing skill name,
- whether each skill ID maps consistently to one known name,
- whether missing names can be recovered from other interactions with the same skill ID,
- and whether any skill IDs have no known name anywhere in the dataset.

`skill_id` remains the primary concept identifier regardless of whether a human-readable name is available.

In [25]:
missing_skill_name = (
    saved_concept_df["skill_name"].isna()
)

print(
    "Concept interactions:",
    len(saved_concept_df)
)

print(
    "Missing skill_name:",
    missing_skill_name.sum()
)

print(
    "Percentage missing skill_name:",
    round(
        missing_skill_name.mean() * 100,
        2
    ),
    "%"
)

print()


# Known names associated with each skill ID
skill_name_mapping = (
    saved_concept_df
    .dropna(subset=["skill_name"])
    .groupby("skill_id")["skill_name"]
    .agg(
        lambda values:
        sorted(set(values))
    )
)

name_count_per_skill = (
    skill_name_mapping
    .apply(len)
)

print(
    "Skill IDs with at least one known name:",
    len(skill_name_mapping)
)

print(
    "Skill IDs mapping to exactly one known name:",
    (name_count_per_skill == 1).sum()
)

print(
    "Skill IDs mapping to multiple known names:",
    (name_count_per_skill > 1).sum()
)

print()


all_skill_ids = set(
    saved_concept_df["skill_id"].unique()
)

skill_ids_with_known_name = set(
    skill_name_mapping.index
)

skill_ids_without_any_name = (
    all_skill_ids
    - skill_ids_with_known_name
)

print(
    "Total unique skill IDs:",
    len(all_skill_ids)
)

print(
    "Skill IDs with no known name anywhere:",
    len(skill_ids_without_any_name)
)

print(
    "IDs with no recoverable name:",
    sorted(skill_ids_without_any_name)
)

print()


recoverable_missing_rows = (
    saved_concept_df.loc[
        missing_skill_name,
        "skill_id"
    ]
    .isin(skill_ids_with_known_name)
)

print(
    "Missing-name interactions recoverable from skill_id mapping:",
    recoverable_missing_rows.sum()
)

print(
    "Missing-name interactions not recoverable:",
    (~recoverable_missing_rows).sum()
)

Concept interactions: 338001
Missing skill_name: 12364
Percentage missing skill_name: 3.66 %

Skill IDs with at least one known name: 111
Skill IDs mapping to exactly one known name: 111
Skill IDs mapping to multiple known names: 0

Total unique skill IDs: 123
Skill IDs with no known name anywhere: 12
IDs with no recoverable name: [np.int64(37), np.int64(76), np.int64(91), np.int64(94), np.int64(96), np.int64(97), np.int64(99), np.int64(101), np.int64(102), np.int64(104), np.int64(105), np.int64(367)]

Missing-name interactions recoverable from skill_id mapping: 0
Missing-name interactions not recoverable: 12364


### 6.2 Skill-Name Handling Decision

The concept-metadata investigation produced the following findings:

- The Concept Interaction Dataset contains 123 unique `skill_id` values.
- `skill_id` is available for every Concept Interaction.
- 111 skill IDs have at least one known `skill_name`.
- Each of these 111 skill IDs maps consistently to exactly one known name.
- No skill ID maps to multiple conflicting names.
- 12 skill IDs have no known `skill_name` anywhere in the dataset.
- These IDs are: 37, 76, 91, 94, 96, 97, 99, 101, 102, 104, 105, and 367.
- The 12,364 Concept Interactions with missing `skill_name` belong to these unrecoverable skill IDs.
- Therefore, none of the missing names can be reliably reconstructed from another interaction in this dataset.

Based on this evidence:

1. `skill_id` will remain the primary and authoritative concept identifier.
2. Existing `skill_name` values will be preserved as human-readable metadata.
3. Missing `skill_name` values will remain unavailable rather than being guessed or artificially imputed.
4. Concept histories and learning-state calculations will not depend on the presence of `skill_name`.
5. APIs or user interfaces may display the human-readable name when available and fall back to the `skill_id` when no name is available.

This preserves concept information without introducing unsupported metadata.

### 6.3 Investigation of `bottom_hint`

The dataset contains a `bottom_hint` field, but a large proportion of its values are missing.

Before using this variable as a learning-behaviour feature, its relationship with hint usage must be investigated.

The analysis checks whether `bottom_hint` is primarily recorded when hints are used, whether missing values represent a meaningful structural condition, and whether this field provides useful information beyond `hint_count` and `hint_total`.

No missing `bottom_hint` values are imputed during this investigation.

In [26]:
print("bottom_hint summary:")

print(
    "Total Base Interactions:",
    len(saved_base_df)
)

print(
    "Missing bottom_hint:",
    saved_base_df["bottom_hint"].isna().sum()
)

print(
    "Percentage missing:",
    round(
        saved_base_df["bottom_hint"].isna().mean()
        * 100,
        2
    ),
    "%"
)

print()

print("Observed bottom_hint values:")
print(
    saved_base_df["bottom_hint"]
    .value_counts(dropna=False)
)

print()

print("bottom_hint by hint usage:")
print(
    pd.crosstab(
        saved_base_df["hint_count"] > 0,
        saved_base_df["bottom_hint"],
        dropna=False,
        margins=True
    )
)

print()

print("Missing bottom_hint by hint usage:")

bottom_hint_missing_by_usage = (
    saved_base_df
    .groupby(
        saved_base_df["hint_count"] > 0
    )["bottom_hint"]
    .apply(
        lambda x: x.isna().mean() * 100
    )
    .round(2)
)

print(bottom_hint_missing_by_usage)

print()

hint_users = saved_base_df[
    saved_base_df["hint_count"] > 0
]

print(
    "Interactions using hints:",
    len(hint_users)
)

print(
    "Hint-using interactions with bottom_hint available:",
    hint_users["bottom_hint"].notna().sum()
)

print(
    "Hint-using interactions with bottom_hint missing:",
    hint_users["bottom_hint"].isna().sum()
)

print()

print("For interactions using hints:")
print(
    hint_users["bottom_hint"]
    .value_counts(dropna=False)
)

bottom_hint summary:
Total Base Interactions: 346860
Missing bottom_hint: 287003
Percentage missing: 82.74 %

Observed bottom_hint values:
bottom_hint
NaN    287003
1.0     42932
0.0     16925
Name: count, dtype: int64

bottom_hint by hint usage:
bottom_hint    0.0    1.0     NaN     All
hint_count                               
False            3      0  287003  287006
True         16922  42932       0   59854
All          16925  42932  287003  346860

Missing bottom_hint by hint usage:
hint_count
False    100.0
True       0.0
Name: bottom_hint, dtype: float64

Interactions using hints: 59854
Hint-using interactions with bottom_hint available: 59854
Hint-using interactions with bottom_hint missing: 0

For interactions using hints:
bottom_hint
1.0    42932
0.0    16922
Name: count, dtype: int64


### 6.4 Behaviour Associated with `bottom_hint`

The missingness analysis shows that `bottom_hint` is structurally associated with hint usage.

When hints are not used, `bottom_hint` is generally unavailable. When hints are used, `bottom_hint` is recorded for every interaction.

Therefore, missing `bottom_hint` values should not automatically be interpreted as `0`.

To determine whether the field provides additional behavioural information, interactions that used hints are examined separately according to whether `bottom_hint = 0` or `bottom_hint = 1`.

The comparison focuses on hint usage, attempt behaviour, correctness, and response time.

In [27]:
bottom_hint_analysis = (
    saved_base_df[
        (saved_base_df["hint_count"] > 0)
        &
        (saved_base_df["bottom_hint"].notna())
    ]
    .copy()
)

bottom_hint_analysis["hint_usage_rate"] = (
    bottom_hint_analysis["hint_count"]
    / bottom_hint_analysis["hint_total"]
)

bottom_hint_summary = (
    bottom_hint_analysis
    .groupby("bottom_hint")
    .agg(
        interactions=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_hint_count=("hint_count", "mean"),
        avg_hint_usage_rate=("hint_usage_rate", "mean"),
        avg_attempts=("attempt_count", "mean"),
        median_attempts=("attempt_count", "median"),
        multi_attempt_rate=(
            "attempt_count",
            lambda x: (x > 1).mean()
        ),
        median_response_ms=(
            "ms_first_response",
            lambda x: x[x > 0].median()
        )
    )
    .round(3)
)

print(
    "Hint-using interactions analysed:",
    len(bottom_hint_analysis)
)

print()

print("Behaviour by bottom_hint:")
print(
    bottom_hint_summary.to_string()
)

print()

print("bottom_hint distribution among hint users:")
print(
    bottom_hint_analysis["bottom_hint"]
    .value_counts()
    .sort_index()
)

print()

print(
    "Percentage reaching bottom_hint:",
    round(
        (
            bottom_hint_analysis["bottom_hint"]
            == 1
        ).mean() * 100,
        2
    ),
    "%"
)

Hint-using interactions analysed: 59854

Behaviour by bottom_hint:
             interactions  accuracy  avg_hint_count  avg_hint_usage_rate  avg_attempts  median_attempts  multi_attempt_rate  median_response_ms
bottom_hint                                                                                                                                    
0.0                 16922     0.001           2.139                0.554         3.267              2.0               0.644             20772.0
1.0                 42932     0.000           3.275                0.994         3.254              2.0               0.544             13189.0

bottom_hint distribution among hint users:
bottom_hint
0.0    16922
1.0    42932
Name: count, dtype: int64

Percentage reaching bottom_hint: 71.73 %


### 6.5 `bottom_hint` Handling Decision

The `bottom_hint` investigation produced the following findings:

- 287,003 Base Interactions (82.74%) have a missing `bottom_hint`.
- This missingness is strongly structural rather than random.
- Among 287,006 interactions where no hints were used, 287,003 have a missing `bottom_hint`.
- Among all 59,854 interactions where at least one hint was used, `bottom_hint` is available.
- Therefore, a missing `bottom_hint` generally represents a situation where the field is not applicable because hint usage did not occur, rather than an ordinary missing measurement.
- Among hint-using interactions, 42,932 (71.73%) have `bottom_hint = 1`, while 16,922 have `bottom_hint = 0`.
- Interactions with `bottom_hint = 1` have an average normalized hint-usage rate of approximately 0.994, compared with 0.554 for `bottom_hint = 0`.
- This indicates that `bottom_hint` contains additional information about progression through the available hints.

Based on this evidence:

1. Original `bottom_hint` values will be preserved.
2. Missing `bottom_hint` values will not be automatically replaced with `0`.
3. `bottom_hint` will be interpreted only in the context of interactions where hints were used.
4. Future feature engineering may distinguish between:
   - no hint usage / `bottom_hint` not applicable,
   - hints used without reaching the bottom hint,
   - hints used and the bottom hint reached.
5. `bottom_hint` may be used as supporting difficulty/help-seeking evidence, but it will not replace the more general `hint_count` and normalized `hint_usage_rate` features.

This treatment preserves the structural meaning of the field without incorrectly interpreting non-applicable values as negative observations.

## 7. Final Learning-Signal Quality Checks

### 7.1 Missing-Value Audit of Candidate Learning Signals

Before feature engineering begins, the processed datasets are checked for missing values in the behavioural and identity fields that may contribute to the student learning-state representation.

This audit distinguishes between:

- required interaction identifiers,
- core behavioural signals,
- concept identifiers,
- and fields where missingness has already been shown to be structurally meaningful.

The purpose is to ensure that unexpected missing values are identified before historical student features are constructed.

In [28]:
base_candidate_fields = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "correct",
    "attempt_count",
    "hint_count",
    "hint_total",
    "ms_first_response",
    "first_action",
    "bottom_hint"
]

concept_candidate_fields = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "correct",
    "attempt_count",
    "hint_count",
    "hint_total",
    "ms_first_response",
    "first_action",
    "bottom_hint"
]


def missing_value_audit(df, fields):

    rows = []

    for field in fields:

        missing = df[field].isna().sum()

        rows.append(
            {
                "field": field,
                "missing": missing,
                "percentage": round(
                    missing / len(df) * 100,
                    3
                )
            }
        )

    return pd.DataFrame(rows)


base_missing_audit = missing_value_audit(
    saved_base_df,
    base_candidate_fields
)

concept_missing_audit = missing_value_audit(
    saved_concept_df,
    concept_candidate_fields
)


print("BASE INTERACTION CANDIDATE SIGNALS")
print(
    base_missing_audit.to_string(
        index=False
    )
)

print()

print("CONCEPT INTERACTION CANDIDATE SIGNALS")
print(
    concept_missing_audit.to_string(
        index=False
    )
)

print()

base_unexpected_fields = [
    field
    for field in base_candidate_fields
    if field != "bottom_hint"
]

concept_unexpected_fields = [
    field
    for field in concept_candidate_fields
    if field not in [
        "bottom_hint",
        "skill_name"
    ]
]

print(
    "Unexpected missing values in required Base fields:",
    saved_base_df[
        base_unexpected_fields
    ].isna().sum().sum()
)

print(
    "Unexpected missing values in required Concept fields:",
    saved_concept_df[
        concept_unexpected_fields
    ].isna().sum().sum()
)

BASE INTERACTION CANDIDATE SIGNALS
            field  missing  percentage
         order_id        0       0.000
          user_id        0       0.000
    assignment_id        0       0.000
       problem_id        0       0.000
          correct        0       0.000
    attempt_count        0       0.000
       hint_count        0       0.000
       hint_total        0       0.000
ms_first_response        0       0.000
     first_action        0       0.000
      bottom_hint   287003      82.743

CONCEPT INTERACTION CANDIDATE SIGNALS
            field  missing  percentage
         order_id        0       0.000
          user_id        0       0.000
    assignment_id        0       0.000
       problem_id        0       0.000
         skill_id        0       0.000
       skill_name    12364       3.658
          correct        0       0.000
    attempt_count        0       0.000
       hint_count        0       0.000
       hint_total        0       0.000
ms_first_response        0   

### 7.2 Candidate Learning-Signal Domain Validation

After checking missing values, the candidate learning signals are validated for their basic value domains.

This check verifies that:

- correctness remains binary,
- attempt and hint counts are non-negative,
- `hint_count` does not exceed `hint_total`,
- `first_action` contains only the observed encoded categories,
- `bottom_hint` contains only its expected observed binary values when available,
- and concept identifiers remain positive.

Response-time values are not required to be positive in this validation because non-positive response times were already separately investigated and intentionally preserved as raw evidence.

The purpose of this check is to detect accidental corruption introduced during preprocessing.

In [29]:
domain_checks = {
    "correct contains only 0/1":
        saved_base_df["correct"].isin([0, 1]).all(),

    "attempt_count is non-negative":
        (saved_base_df["attempt_count"] >= 0).all(),

    "hint_count is non-negative":
        (saved_base_df["hint_count"] >= 0).all(),

    "hint_total is non-negative":
        (saved_base_df["hint_total"] >= 0).all(),

    "hint_count <= hint_total":
        (
            saved_base_df["hint_count"]
            <= saved_base_df["hint_total"]
        ).all(),

    "first_action contains only observed codes 0/1/2":
        saved_base_df["first_action"]
        .isin([0, 1, 2])
        .all(),

    "bottom_hint contains only 0/1 when available":
        saved_base_df["bottom_hint"]
        .dropna()
        .isin([0, 1])
        .all(),

    "concept skill_id is positive":
        (saved_concept_df["skill_id"] > 0).all(),

    "concept raw_row_count >= 1":
        (saved_concept_df["raw_row_count"] >= 1).all(),

    "opportunity_end >= opportunity_start":
        (
            saved_concept_df["opportunity_end"]
            >= saved_concept_df["opportunity_start"]
        ).all()
}

domain_check_table = pd.DataFrame(
    [
        {
            "check": check,
            "passed": passed
        }
        for check, passed
        in domain_checks.items()
    ]
)

print("Candidate learning-signal domain checks:")
print(
    domain_check_table.to_string(
        index=False
    )
)

print()

print(
    "Checks passed:",
    sum(domain_checks.values()),
    "/",
    len(domain_checks)
)

print(
    "All domain checks passed:",
    all(domain_checks.values())
)

Candidate learning-signal domain checks:
                                          check  passed
                      correct contains only 0/1    True
                  attempt_count is non-negative    True
                     hint_count is non-negative    True
                     hint_total is non-negative    True
                       hint_count <= hint_total    True
first_action contains only observed codes 0/1/2    True
   bottom_hint contains only 0/1 when available    True
                   concept skill_id is positive    True
                     concept raw_row_count >= 1    True
           opportunity_end >= opportunity_start    True

Checks passed: 10 / 10
All domain checks passed: True


## 8. Final Preprocessing Integrity Validation

Before the processed interaction datasets are accepted for downstream feature engineering, a final consolidated integrity validation is performed.

This validation combines the major structural requirements established during preprocessing.

The Base Interaction Dataset must:

- contain exactly one row per `order_id`,
- preserve all students and interactions including those without known skills,
- contain no unexpected missing values in required learning-signal fields.

The Concept Interaction Dataset must:

- contain exactly one row per `order_id + skill_id`,
- contain no missing concept identifiers,
- preserve valid opportunity ranges,
- preserve the complete known-skill mapping from the raw data,
- and contain no unexpected missing values in required learning-signal fields.

The purpose of this final check is to establish a clear preprocessing completion criterion before feature engineering begins.

In [30]:
final_checks = {
    # Base interaction integrity
    "Base rows = 346860":
        len(saved_base_df) == 346860,

    "Base order_id unique":
        saved_base_df["order_id"].is_unique,

    "Base students = 4217":
        saved_base_df["user_id"].nunique() == 4217,

    "Base required fields have no unexpected missing values":
        saved_base_df[
            [
                "order_id",
                "user_id",
                "assignment_id",
                "problem_id",
                "correct",
                "attempt_count",
                "hint_count",
                "hint_total",
                "ms_first_response",
                "first_action"
            ]
        ].isna().sum().sum() == 0,

    # Concept interaction integrity
    "Concept rows = 338001":
        len(saved_concept_df) == 338001,

    "Concept order_id + skill_id unique":
        ~saved_concept_df.duplicated(
            subset=["order_id", "skill_id"]
        ).any(),

    "Concept students = 4163":
        saved_concept_df["user_id"].nunique() == 4163,

    "Concept skill_id complete":
        saved_concept_df["skill_id"].notna().all(),

    "Concept skill_id integer dtype":
        str(saved_concept_df["skill_id"].dtype) == "int64",

    "Concept opportunity ranges valid":
        (
            saved_concept_df["opportunity_end"]
            >= saved_concept_df["opportunity_start"]
        ).all(),

    "Concept raw_row_count >= 1":
        (
            saved_concept_df["raw_row_count"] >= 1
        ).all(),

    "Concept required fields have no unexpected missing values":
        saved_concept_df[
            [
                "order_id",
                "user_id",
                "assignment_id",
                "problem_id",
                "skill_id",
                "correct",
                "attempt_count",
                "hint_count",
                "hint_total",
                "ms_first_response",
                "first_action"
            ]
        ].isna().sum().sum() == 0
}


final_validation_table = pd.DataFrame(
    [
        {
            "validation": name,
            "passed": result
        }
        for name, result in final_checks.items()
    ]
)


print("FINAL PREPROCESSING INTEGRITY VALIDATION")
print("=" * 60)

print(
    final_validation_table.to_string(
        index=False
    )
)

print()
print(
    "Checks passed:",
    sum(final_checks.values()),
    "/",
    len(final_checks)
)

print(
    "FINAL RESULT:",
    (
        "PASS"
        if all(final_checks.values())
        else "FAIL"
    )
)

FINAL PREPROCESSING INTEGRITY VALIDATION
                                               validation  passed
                                       Base rows = 346860    True
                                     Base order_id unique    True
                                     Base students = 4217    True
   Base required fields have no unexpected missing values    True
                                    Concept rows = 338001    True
                       Concept order_id + skill_id unique    True
                                  Concept students = 4163    True
                                Concept skill_id complete    True
                           Concept skill_id integer dtype    True
                         Concept opportunity ranges valid    True
                               Concept raw_row_count >= 1    True
Concept required fields have no unexpected missing values    True

Checks passed: 12 / 12
FINAL RESULT: PASS


## 9. Preprocessing Phase Conclusion

The preprocessing phase has been completed successfully.

The original ASSISTments Skill Builder dataset contained 525,534 raw rows. Investigation showed that these rows do not directly correspond to independent student interactions because the raw representation contains repeated opportunity rows and interactions associated with multiple skills.

Two validated interaction-level datasets were therefore constructed.

### Base Interaction Dataset

- 346,860 Base Interactions
- 4,217 students
- exactly one row per `order_id`
- no duplicate `order_id`
- interactions without known skills are preserved
- core behavioural evidence remains available for student-level history construction

### Concept Interaction Dataset

- 338,001 Concept Interactions
- 4,163 students with known-skill interactions
- 123 unique skill IDs
- exactly one row per `order_id + skill_id`
- no missing `skill_id`
- multi-skill interactions are preserved
- repeated opportunity rows are represented using validated opportunity ranges and `raw_row_count`

### Data-Preservation Validation

The processed Base Interaction values exactly match their corresponding raw interaction values.

The processed Concept Interaction values also exactly match their corresponding raw concept-level reference values.

Therefore, preprocessing changes the representation of the raw dataset without silently changing the underlying behavioural evidence.

### Behavioural Data-Quality Decisions

The following evidence-based decisions were established:

- Zero attempt counts are preserved because they are strongly associated with structured `first_action` behaviour.
- Extreme attempt counts are preserved as potential difficulty evidence, but raw unbounded averages will not be relied upon during feature engineering.
- Non-positive response times are preserved in the stored interaction data but will be treated as unavailable timing evidence when response-time features are constructed.
- Extreme positive response times are preserved, while robust response-time representations will be used during feature engineering.
- `hint_count` and `hint_total` are preserved.
- Normalized hint usage may be calculated only when `hint_total > 0`.
- `hint_total = 0` is distinguished from situations where hints were available but unused.
- Missing `bottom_hint` values are treated as structurally non-applicable rather than automatically replaced with zero.
- `skill_id` is the authoritative concept identifier.
- Missing `skill_name` values for 12 skill IDs are preserved as unavailable rather than guessed.

### Final Integrity Result

The final preprocessing integrity validation passed all 12 defined checks.

**Final Result: PASS**

The processed datasets are therefore accepted as the validated input foundation for the next development phase: historical feature engineering for student learning-state identification.